
# Refactored ECEarth Susceptibility Builder

This refactored notebook focuses on:
- Performance: avoid giant Dask graphs; save per-station outputs; use `xarray.open_mfdataset`
- Robustness: handle serialization limits and worker memory; persist at sensible points
- Readability: modular functions, clear variable naming, and comments
- Scientific correctness: single global 34-level pressure grid and consistent interpolation

**Files produced by this notebook** will be saved under `/mnt/data/ccn_cdnc_per_station/` by default.


In [1]:

# Prerequisites and imports
import os
import xarray as xr
import numpy as np
import dask
from dask import delayed
from dask.distributed import Client, wait, performance_report
import gc

# Configure Dask client (adjust n_workers, threads_per_worker, memory_limit as appropriate)
client = Client()  # modify arguments if you run on a cluster
print(client)


<Client: 'tcp://127.0.0.1:40313' processes=8 threads=32, memory=125.79 GiB>


2025-10-30 16:29:36,362 - distributed.scheduler - WARNING - Detected different `run_spec` for key 'original-open_dataset-var54-225b623b7476a239322994895db6ddef' between two consecutive calls to `update_graph`. This can cause failures and deadlocks down the line. Please ensure unique key names. If you are using a standard dask collections, consider releasing all the data before resubmitting another computation. More details and help can be found at https://github.com/dask/dask/issues/9888. 
Debugging information
---------------------
old task state: memory
old run_spec: (<function execute_task at 0x7fb7ccd03420>, (ImplicitToExplicitIndexingAdapter(array=CopyOnWriteArray(array=LazilyIndexedArray(array=<xarray.backends.netCDF4_.NetCDF4ArrayWrapper object at 0x7fb750551ac0>, key=BasicIndexer((slice(None, None, None), slice(None, None, None), slice(None, None, None), slice(None, None, None)))))),), {})
new run_spec: (<function execute_task at 0x7fb7ccd03420>, (ImplicitToExplicitIndexingAdap

In [2]:

# Configuration
ECPath = "/share/sabl0586/all_stations_EC-Earth_PRCP2SZDST_ilevall_levs_4Peter.nc"   # <-- change to your path or keep using your extractor func
ds = xr.open_dataset(ECPath, chunks={})
stations = ds["station"].values  # list of station ids
radii = np.arange(20,51)  # example; replace with your radii array

output_dir = "/share/pech2273/PhD/EC-Earth"
os.makedirs(output_dir, exist_ok=True)


In [3]:

# --- Compute global mean pressure levels (34 levels expected) ---
# Open one representative dataset lazily and compute mean over time and station.
# Adjust variable names/path to match your files or the output of ECearthExtract_Dask.
ds_meta = xr.open_dataset(ECPath, engine="netcdf4", chunks="auto")  # lazy
# Ensure the pressure variable name matches; change "pressure" if needed
if 'pressure' not in ds_meta.variables:
    raise KeyError("Expected 'pressure' variable in dataset. Inspect your dataset variables: " + ", ".join(ds_meta.variables.keys()))

# Compute mean across time & station to get target vertical grid (lazy -> compute once)
Pressure_mean_levs = ds_meta['pressure'].mean(dim=['time', 'station']).compute()
# If your pressure is in Pa and you want hPa levels:
Pressure_mean_levs_hPa = (Pressure_mean_levs / 100).astype(float)
Pressure_mean_levs_hPa.attrs['units'] = 'hPa'
print("Computed target vertical levels:", Pressure_mean_levs_hPa.shape)


Computed target vertical levels: (34,)


In [4]:

# --- Per-station processing (delayed) ---
# This function is intentionally self-contained. It loads station data (via your extractor),
# interpolates to the shared Pressure_mean_levs, computes CDNC and CCN, and writes netCDF to disk.
@delayed
def process_station_write(station, ECPath, output_dir, radii, target_lev_pa):
    import xarray as xr
    import numpy as np
    # Replace ECearthExtract_Dask with your actual function that returns (EC_ds, ds_ifs)
    EC_ds, ds_ifs = ECearthExtract_Dask(ECPath, station, chunks='auto')
    
    # Prepare IFS dataset for interpolation
    ds_IFS = ds_ifs.rename({'lev_ifs': 'lev'}).assign_coords(lev=ds_ifs['lev'])
    # Interpolate IFS vars to the target lev (in Pa)
    ds_ifs_interp = ds_IFS.interp(lev=target_lev_pa, method='linear', kwargs={'fill_value': 'extrapolate'})
    
    # Compute CDNC safely
    cdnc = (ds_ifs_interp['var20'] / ds_ifs_interp['var22']).where(ds_ifs_interp['var22'] > 0)
    cdnc = cdnc.assign_coords(lev=(target_lev_pa / 100.0))
    cdnc['lev'].attrs['units'] = 'hPa'
    
    # Compute CCN using your function (must accept EC_ds and radii)
    CCN_ds = Function.ECEarthERF(EC_ds, radii)
    # Ensure CCN_ds lev coord converted to hPa and matches target
    CCN_ds = CCN_ds.assign_coords(lev=(target_lev_pa / 100.0))
    CCN_ds['lev'].attrs['units'] = 'hPa'
    
    # Align and build final dataset
    CCN_aligned, CDNC_aligned = xr.align(CCN_ds, cdnc, join='inner')
    ds_out = xr.Dataset(
        data_vars={ 'CCN': CCN_aligned, 'CDNC': CDNC_aligned },
        coords={ 'radius': CCN_aligned.radius, 'lev': CCN_aligned.lev, 'time': CCN_aligned.time }
    )
    ds_out = ds_out.assign_coords(station=station)
    
    # Write to single-file netCDF for this station (use netCDF4 or zarr if preferred)
    outfile = os.path.join(output_dir, f"CCN_CDNC_{station}.nc")
    encoding = {v: {'zlib': True, 'complevel': 4} for v in ds_out.data_vars}
    ds_out.to_netcdf(outfile, format='NETCDF4', encoding=encoding)
    return outfile


In [5]:
def ECearthExtract_Dask(ECPath, station, chunks="auto"):
    """
    Lazily load EC-Earth data using Dask and optionally compute PNSD.

    Parameters
    ----------
    ECPath : str
        Path to EC-Earth NetCDF file.
    station : str
        Station name or coordinate.
    chunks : str or dict, optional
        Dask chunking specification. Default 'auto'.
    """

    import xarray as xr

    #Open lazily with Dask
    Data = xr.open_dataset(ECPath, chunks=chunks)
    Data = Data.sel(station=station)

    ds = xr.Dataset()
    PNSD_ds = xr.Dataset()
    cdnc = xr.Dataset()
    ds_IFS = xr.Dataset()


    #Define variables for the ERF
    radius_variables = ['RDRY_NUS', 'RDRY_AIS', 'RDRY_ACS', 'RWET_AII', 'RDRY_COS', 'RWET_ACI', 'RWET_COI']
    Numb_variables = ['N_NUS', 'N_AIS', 'N_ACS', 'N_AII', 'N_COS', 'N_ACI', 'N_COI']
    ModesSigma = [1.59, 1.59, 1.59, 2.0, 1.59, 1.59, 2.0]
    
    for radius, number in zip(radius_variables, Numb_variables):
        if radius in Data and number in Data:
            ds[radius] = Data[radius]*1e9
            ds[radius].attrs["units"] = "nm"
            #print(f' {radius} added to Dataset')
            ds[number] = Data[number]/1e6
            ds[number].attrs["units"] = "#cm-3"
            #print(f' {number} added to Dataset')
        else:
            print(f'{radius, number} not found in EC Path Data')
            
    # ADD PRESSURE TO DATASET        
    ds['pressure'] = Data['pressure']

    #Converting the IFS levels to match the EC-Earth Levs

    for ifs in [['var20', 'var22', 'var54']]:
        ds_IFS[ifs] = Data[ifs]
    ds_IFS = ds_IFS[['var20', 'var22', 'var54']].isel(lev=0).drop_vars('lev')
    ds_IFS['lev_ifs'] = ds_IFS['var54'].mean('time')

    
    return ds, ds_IFS

In [6]:

# Kick off delayed processing for all stations
target_lev_pa = (Pressure_mean_levs_hPa * 100).values  # Pa values for interpolation
futures = [process_station_write(s, ECPath, output_dir, radii, target_lev_pa) for s in stations]

# Compute with Dask: returns list of file paths
results = dask.compute(*futures)
print("Written files:", results)

# List files
import glob
files = sorted(glob.glob(os.path.join(output_dir, 'CCN_CDNC_*.nc')))
print(f"Found {len(files)} station files.")


2025-10-30 16:29:37,730 - distributed.worker - ERROR - Compute Failed
Key:       process_station_write-3aff0f9c-37c8-4246-8a31-39b9363520b2
State:     long-running
Function:  process_station_write
args:      (np.str_('SMR-II'), '/share/sabl0586/all_stations_EC-Earth_PRCP2SZDST_ilevall_levs_4Peter.nc', '/share/pech2273/PhD/EC-Earth', array([20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]), array([9.60637878e+04, 9.51944824e+04, 9.32441589e+04, 8.94417419e+04,
       8.51861938e+04, 7.90531433e+04, 7.20771790e+04, 6.42852112e+04,
       5.63022949e+04, 4.83550354e+04, 4.06364227e+04, 3.43126007e+04,
       2.96059662e+04, 2.62490784e+04, 2.30728638e+04, 2.04146057e+04,
       1.81000702e+04, 1.56608353e+04, 1.39999161e+04, 1.22485847e+04,
       1.06517502e+04, 9.21759109e+03, 8.03970718e+03, 6.70043564e+03,
       5.41842880e+03, 4.03461456e+03, 2.81728840e+03, 1.88720131e+03,
       1.15508842e+03, 6.471

KeyError: "No variable named 'lev'. Did you mean one of ('lev_ifs',)?"

In [ ]:

# Reopen the per-station NetCDF files lazily
CCN_CDNC_all_ds = xr.open_mfdataset(
    os.path.join(output_dir, 'CCN_CDNC_*.nc'),
    combine='nested',
    concat_dim='station',
    chunks={'time': 500, 'lev': -1, 'radius': -1}
)

print(CCN_CDNC_all_ds)
# Persist small parts if needed for plotting
# For plotting a small slice, compute only the selection:
subset = CCN_CDNC_all_ds['CCN'].sel(radius=35, station='SMR-II').compute()
subset.plot()



## Diagnostics & Best Practices

- Use `client.restart()` to clear the cluster state if you hit serialization issues.
- Prefer **writing per-station outputs** to disk (NetCDF/Zarr) then reopen with `open_mfdataset`.
- If memory is tight: use Zarr store (chunked, cloud-friendly) instead of single-file NetCDF.
- When chunking: avoid huge serialized graphs; keep chunk sizes moderate (e.g., time: 500, lev: 20).
- If you need to tune Dask serialization limits, consider `distributed.comm.tcp` settings, but prefer reducing graph size.
- For reproducibility, record the Dask client config with `client.get_worker_logs()` and `client.scheduler_info()`.
